# Imports

In [ ]:
import torch
import pandas as pd
from pathlib import Path
import gc
from tqdm.auto import tqdm
import warnings
import pingouin as pg
from sentence_transformers import SentenceTransformer, util
from pathlib import Path
import json, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np

device = 0 if torch.cuda.is_available() else -1
print(device)

In [ ]:
!pip install pingouin

In [ ]:
!pip install qlatent

In [ ]:
from qlatent.qmnli.qmnli import *
from qlatent.qmnli.qmnli import _QMNLI, QMNLI


In [ ]:
# softmax_files = [False, True]
softmax_files = [True]

def split_question(Q, index, scales, softmax, filters):
  result = []
  for s in scales:
    q = QCACHE(Q(index=index, scale=s))
    for sf in softmax:
      for f in filters:
        if sf:
            qsf = QSOFTMAX(q,dim=[index[0], s])
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print((index, s),sf,f)
            result.append(qsf_f)
        else:
            qsf = QPASS(q,descupdate={'softmax':''})
            qsf_f = QFILTER(qsf,filters[f],filtername=f)
            print(s,sf,f)
            result.append(qsf_f)
  return result


def print_permutations(q):
#     for q in Q1s:
    W = q._pdf['W']
    print(q._descriptor)
    for i, (kmap, w) in enumerate(zip(q._keywords_map, W)):
        context = q._context_template.format_map(kmap)
        answer = q._answer_template.format_map(kmap)
#         sexisem_score = sexisem_classifier(context.strip('.') + ' ' +answer)
        print(f'{i}.',context ,'->', answer, w)
#     break


frequency_weights:SCALE = {
    'never':-4,
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'frequently':2,
    'often':2,
    'very frequently':3,
    'always':4,
}

intensifiers_fraction_without_none:SCALE={
            "few":1,
            "some":2,
            "many":3,
            "most":4,
            "all":5,
        }

full_frequency_weights = {
    'never':-4,
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'usually':1,
    'frequently':2,
    'often':2,
    'very frequently':3,
    'always':4,
}

partial_frequency_weights0 = {
    'never':-4,
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'frequently':2,
    'often':2,
    'very frequently':3,
    'always':4,
}
partial_frequency_weights1 = {
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'usually':1,
    'frequently':2,
    'often':2,
    'very frequently':3,
}
partial_frequency_weights2 = {
    'very rarely':-3,
    'seldom':-2,
    'rarely':-2,
    'frequently':2,
    'often':2,
    'very frequently':3,
}


partial_frequency_weights3 = {
    'seldom':-2,
    'rarely':-2,
    'usually':1,
    'often':2,
    'frequently':2
}
partial_frequency_weights4 = {
    'seldom':-2,
    'rarely':-2,
    'usually':1,
    'often':2,
}


agreement_weights = {
    "strongly agree" : 2,
    "agree" : 1,
    "am neutral" : 0,
    "disagree" : -1,
    "strongly disagree" : -2
}

In [ ]:
# p = 'valhalla/distilbart-mnli-12-6'
p = "typeform/distilbert-base-uncased-mnli"
mnli = pipeline("zero-shot-classification",device=device, model=p)
mnli.model_identifier = p

In [ ]:
kw_long_male_plural = ["men","boys","males"]
kw_long_male_singular = ["man","boy","male"]
kw_long_female_plural = ["women","girls","females"]
kw_long_female_singular = ["woman","girl","female"]
kw_long_genderneutral_plural = ["people", "folks", "community members", "citizen"]
kw_short_male_plural = ["men"]
kw_short_male_singular = ["man"]
kw_short_female_plural = ["women"]
kw_short_female_singular = ["woman"]

kw_long_genderboth_plural = ["men and women","women and men","females and males", "males and females", "boys and girls","girls and boys"]

dict_long_gender_plural      = dict(dict_same_weight(1,kw_long_male_plural),    **dict_same_weight(1,kw_long_female_plural))
dict_long_gender_singluar    = dict(dict_same_weight(1,kw_long_male_singular),  **dict_same_weight(1,kw_long_female_singular))
dict_short_gender_plural     = dict(dict_same_weight(1,kw_short_male_plural),   **dict_same_weight(1,kw_short_female_plural))
dict_short_gender_singular   = dict(dict_same_weight(1,kw_short_male_singular), **dict_same_weight(1,kw_short_female_singular))

In [ ]:
softmax_files = [True, False]

# Linguastic acceptability

In [ ]:
sentence_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
cola = pipeline("text-classification","mrm8488/deberta-v3-small-finetuned-cola", device=device)

import os
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu

def linguistic_acceptabilities(q, index, scale,question_name, student_id, output_path=Path(''), save_to_file=False):
    score_by_cola_lst=[]
    score_of_semantic_distance_lst=[]
    score_by_bleu_lst=[]
    kmap_lst=[]
    question_name_lst=[]
    description = q._descriptor
    strFactor=description['Factor']
    strOrdinal=str(description.get('Ordinal', 0))
    ##cleaning the string to get the original question
    strOriginal= description['Original']
    strOriginal = 'none' if strOriginal is None else strOriginal
    strOriginal=strOriginal.replace(strFactor,'',1)
    strOriginal=strOriginal.replace(strOrdinal,'',1)
    strOriginal=strOriginal.replace('.','',1)
    strOriginal=strOriginal.strip() #the original question
    rows = []
    if hasattr(q, 'linguistic_acceptability'):
        return q.linguistic_acceptability
    partial_internal_consistency = partial(q.internal_consistency, filter={}, index=index , scale=scale)
    try:
        silhouette_score = partial_internal_consistency(measure='silhouette_score', metric='correlation')
    except Exception as e:
        print(e)
        print('silhouette_score is set to -1')
        silhouette_score = -1

    for kmap in q._keywords_map:
        score = {}
        score['question_name'] = question_name
        context = q._context_template.format_map(kmap)
        answer = q._answer_template.format_map(kmap)
        score['original_question'] = strOriginal


        cola_score = cola(context +" "+ answer)[0].get('score')
        score['cola_score'] = cola_score
        score['param'] = kmap
        strPermutation= context +" "+ answer
        # sentences = [context +" "+ answer]
        score['question_permutation'] = strPermutation
        #Compute embedding for both lists
        embeddings1 = sentence_embedding_model.encode(strOriginal, convert_to_tensor=True)
        embeddings2 = sentence_embedding_model.encode(strPermutation, convert_to_tensor=True)

        #Compute cosine-similarities
        cosine_scores = util.cos_sim(embeddings1, embeddings2)
        score['semantic_similarity'] = cosine_scores.item()

        score['silhouette_score'] = silhouette_score
        rows.append(score)


    filename = output_path / 'linguistic_acceptabilities.csv'
    df = pd.DataFrame(rows)
    df['student_id'] = student_id
    df = df[['student_id', 'question_name','original_question', 'param','question_permutation','cola_score','semantic_similarity','silhouette_score']]
    if save_to_file:
        if filename.exists():
            df.to_csv(filename, index=False, header=None, mode='a', encoding='utf-8-sig')
        else:
            df.to_csv(filename, index=False, encoding='utf-8-sig')
#     print(f"Linguistic acceptabilities saved in {filename}")
    q.linguistic_acceptability = df
    return df

# Basic empathy scale (BES)

## Affective empathy

#### BESQ1: My friend’s emotions don’t affect me much.

In [ ]:
# kw_pos = ['great', 'significant', 'tremendous']
# kw_neg = ['little','no']
# kw_pos = ['often', 'usually','frequently','always','very frequently']
# kw_neg = ['seldom', 'rarely','never','very rarely']
# dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ1(QMNLI):
  """
    BESQ1: My friend’s emotions don’t affect me much.
  """
  def __init__(self, **kwargs):
    super().__init__(
        # context_template="My friend's emotions have {index} impact on me.",
        #context_template="My friend's emotions {agreement_scale} have a great impact on me.",
        context_template="I'm {partial_frequency_weights1} deeply influenced by my friend's emotions.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights1":partial_frequency_weights1,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":1,
                      "Original": "My friend's emotions have a great impact on me." #REVERSED
        },
        **kwargs,
    )



qs1 = split_question(BESQ1,
                      index=["partial_frequency_weights1"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ1().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ1', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ2: After being with a friend who is sad about something, I usually feel sad.

In [ ]:
# kw_pos = ['sad', 'upset','miserable']
# kw_neg = ['happy', 'glad']
# kw_pos = ['often', 'usually']
# kw_neg = ['seldom', 'rarely']
# dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ2(QMNLI):
  """
    BES2: After being with a friend who is sad about something, I usually feel sad
  """
  def __init__(self, **kwargs):
    super().__init__(
        # context_template="After being with a friend who is {index}, I feel sad.",
        #context_template="After spending time with a friend who is feeling sad, I {partial_frequency_weights4} feel sad.",
        context_template="After spending time with a friend who is feeling sad, I {partial_frequency_weights4} find my own emotions mirror theirs.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights4":partial_frequency_weights4,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":2,
                      "Original": 'After being with a friend who is sad about something, I usually feel sad.'
        },
        **kwargs,
    )


qs2 = split_question(BESQ2,
                      index=["partial_frequency_weights4"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ2().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ2', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ4: I get frightened when I watch characters in a good scary movie.

In [ ]:
# kw_pos = ['frightened',"afraid","scared"]
# kw_neg = ['comfortable', 'relaxed']
kw_pos = ['often', 'usually','frequently']
kw_neg = ['seldom', 'rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ4(QMNLI):
  """
    BESQ4: I get frightened when I watch characters in a good scary movie.
  """
  def __init__(self, **kwargs):
    super().__init__(
        # context_template="Characters in scary movies make me feel {index}.",
        context_template="Characters in scary movies {partial_frequency_weights3} make me feel frightened.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights3":partial_frequency_weights3,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":4,
                      "Original": 'I get frightened when I watch characters in a good scary movie.'
        },
        **kwargs,
    )



qs4 = split_question(BESQ4,
                      index=["partial_frequency_weights3"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ4().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ4', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ5: I get caught up in other people’s feelings easily.

In [ ]:
# kw_pos = ['easily','always']
# kw_neg = ['hardly', 'barely',"don't"]
kw_pos = ['often','frequently','very frequently']
kw_neg = ['seldom', 'rarely','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ5(QMNLI):
  """
    BESQ5: I get caught up in other people’s feelings easily.
  """
  def __init__(self, **kwargs):
    super().__init__(
        # context_template="People’s feelings {index} affect me.",
        context_template="My sentiments {partial_frequency_weights2} mirror those of the individuals I interact with.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights2":partial_frequency_weights2,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":5,
                      "Original": 'I get caught up in other people’s feelings easily.'
        },
        **kwargs,
    )


qs5 = split_question(BESQ5,
                      index=["partial_frequency_weights2"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ5().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ5', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ7: I don’t become sad when I see other people crying.

In [ ]:
kw_pos = ['often','frequently','very frequently']
kw_neg = ['seldom', 'rarely','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ7(QMNLI):
  """
    BESQ7: I don’t become sad when I see other people crying.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I {partial_frequency_weights3} become sad when I see other people crying.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights3":partial_frequency_weights3,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":7,
                      "Original": 'I become sad when I see other people crying.' #REVERSED
        },
        **kwargs,
    )

qs7 = split_question(BESQ7,
                      index=["partial_frequency_weights3"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ7().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ7', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ8: Other people’s feelings don’t bother me at all.

In [ ]:
# kw_pos = ['bother','annoy','irritate']
# kw_neg = ['please','satisfy']
kw_pos = ['often','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ8(QMNLI):
  """
    BESQ8: Other people’s feelings don’t bother me at all.
  """
  def __init__(self, **kwargs):
    super().__init__(
        # context_template="Other people's feelings {index} me a lot.",
        context_template="The feelings of others {partial_frequency_weights1} bother me greatly.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights1":partial_frequency_weights1,
            "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":8,
                      "Original": "Other people's feelings bother me a lot." #REVERSED
        },
        **kwargs,
    )


qs8 = split_question(BESQ8,
                      index=["partial_frequency_weights1"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ8().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ8', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ11: I often become sad when watching sad things on TV or in films.

In [ ]:
kw_pos = ['often', 'usually']
kw_neg = ['seldom', 'rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ11(QMNLI):
  """
    BESQ11: I often become sad when watching sad things on TV or in films.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I {partial_frequency_weights4} become sad when watching sad things on TV or in films.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights4":partial_frequency_weights4,
          "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":11,
                      "Original": 'I often become sad when watching sad things on TV or in films.'
        },
        **kwargs,
    )


qs11 = split_question(BESQ11,
                      index=["partial_frequency_weights4"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ11().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ11', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ13: Seeing a person who has been angered has no effect on my feelings.

In [ ]:
# kw_pos = ['extremely','incredibly']
# kw_neg = ['barely','hardly']
kw_pos = ['often', 'usually']
kw_neg = ['seldom', 'rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ13(QMNLI):
  """
    BESQ13: Seeing a person who has been angered has no effect on my feelings.
  """
  def __init__(self, **kwargs):
    super().__init__(
        #context_template="My feelings get {index} affected when I see a person get angered.",
        context_template="My feelings {partial_frequency_weights2} get affected when I see a person get angered.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights2":partial_frequency_weights2,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":13,
                      "Original": 'Seeing a person who has been angered affects my feelings.' #REVERSED
        },
        **kwargs,
    )


qs13 = split_question(BESQ13,
                      index=["partial_frequency_weights2"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ13().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ13', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ15: I tend to feel scared when I am with friends who are afraid.

In [ ]:
# kw_pos = ['tend to','usually']
# kw_neg = ['seldom', 'rarely']
kw_pos = ['often', 'usually']
kw_neg = ['seldom', 'rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ15(QMNLI):
  """
    BESQ15: I tend to feel scared when I am with friends who are afraid.
  """
  def __init__(self, **kwargs):
    super().__init__(
        #context_template="I {index} feel scared when I am with friends who are afraid.",
        context_template="I {partial_frequency_weights4} tend to feel scared when I am with friends who are afraid.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights4":partial_frequency_weights4,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":15,
                      "Original": 'I tend to feel scared when I am with friends who are afraid.'
        },
        **kwargs,
    )


qs15 = split_question(BESQ15,
                      index=["partial_frequency_weights4"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ15().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ15', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ17: I often get swept up in my friend’s feelings.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ17(QMNLI):
  """
    BESQ17: I often get swept up in my friend’s feelings.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I {partial_frequency_weights3} get swept up in my friend’s feelings.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights3":partial_frequency_weights3,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":17,
                      "Original": 'I often get swept up in my friend’s feelings.'
        },
        **kwargs,
    )


qs17 = split_question(BESQ17,
                      index=["partial_frequency_weights3"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ17().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ17', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ18: My friend’s unhappiness doesn’t make me feel anything.

In [ ]:
# kw_pos = ['extremely','incredibly']
# kw_neg = ['barely','hardly']
kw_pos = ['often', 'usually']
kw_neg = ['seldom', 'rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ18(QMNLI):
  """
    BESQ18: My friend’s unhappiness doesn’t make me feel anything.
  """
  def __init__(self, **kwargs):
    super().__init__(
        #context_template="My feelings get {index} affected when my friend is unhappy.",
        context_template="My feelings {partial_frequency_weights4} get affected when my friend is unhappy.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights4":partial_frequency_weights4,
                   "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"AE",
                      "Ordinal":18,
                      "Original": 'My friend’s unhappiness affects my feelings.' #REVERSED
        },
        **kwargs,
    )

qs18 = split_question(BESQ18,
                      index=["partial_frequency_weights4"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ18().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },
                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ18', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

## Conginitive empathy

#### BESQ3: I can understand my friend’s happiness when she/he does well at something.

In [ ]:
# kw_pos = ["understand","comprehend"]
# kw_neg = ["am confused by", 'misinterpret']
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ3(QMNLI):
  """
    BESQ3: I can understand my friend’s happiness when she/he does well at something.
  """
  def __init__(self, **kwargs):
    super().__init__(
        #context_template="I {index} my friend's happiness when they succeed.",
        context_template="I {partial_frequency_weights1} can understand my friend's happiness when they succeed.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights1":partial_frequency_weights1,
                  "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":3,
                      "Original": 'I can understand my friend’s happiness when she/he does well at something.'
        },
        **kwargs,
    )



qs3 = split_question(BESQ3,
                      index=["partial_frequency_weights1"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ3().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ3', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ6: I find it hard to know when my friends are frightened.

In [ ]:
# kw_pos = ["complex","challenging"]
# kw_neg = ['easy', 'simple']
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ6(QMNLI):
  """
    BESQ6: I find it hard to know when my friends are frightened.
  """
  def __init__(self, **kwargs):
    super().__init__(
        #context_template="I find it {index} to recognize when my friends are scared.",
        context_template="I {partial_frequency_weights4} find it hard to know when my friends are frightened.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights4":partial_frequency_weights4,
                   "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":6,
                      "Original": 'I find it hard to know when my friends are frightened.'
        },
        **kwargs,
    )



qs6 = split_question(BESQ6,
                      index=["partial_frequency_weights4"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ6().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ6', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ9: When someone is feeling down I can usually understand how she/he feels.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ9(QMNLI):
  """
    BESQ9: When someone is feeling down I can usually understand how she/he feels.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="When someone is feeling down I can {partial_frequency_weights1} understand how she/he feels.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights1":partial_frequency_weights1,
                     "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":9,
                      "Original": 'When someone is feeling down I can usually understand how she/he feels.'
        },
        **kwargs,
    )


qs9 = split_question(BESQ9,
                      index=["partial_frequency_weights1"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ9().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ9', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ10: I can usually work out when my friends are scared.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ10(QMNLI):
  """
    BESQ10: I can usually work out when my friends are scared.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I can {full_frequency_weights} work out when my friends are scared.",
        answer_template="I {agreement_scale}.",
        dimensions={"full_frequency_weights":full_frequency_weights,
                 "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":10,
                      "Original": 'I can usually work out when my friends are scared.'
        },
        **kwargs,
    )


qs10 = split_question(BESQ10,
                      index=["full_frequency_weights"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ10().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ10', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ12: I can often understand how people are feeling even before they tell me.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ12(QMNLI):
  """
    BESQ12: I can often understand how people are feeling even before they tell me.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I can {partial_frequency_weights1} understand how people are feeling even before they tell me.",
        answer_template="I {agreement_scale}.",
        dimensions={"partial_frequency_weights1":partial_frequency_weights1,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":12,
                      "Original": 'I can often understand how people are feeling even before they tell me.'
        },
        **kwargs,
    )


qs12 = split_question(BESQ12,
                      index=["partial_frequency_weights1"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ12().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ12', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ14: I can usually work out when people are cheerful.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ14(QMNLI):
  """
    BESQ14: I can usually work out when people are cheerful.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I can {full_frequency_weights} work out when people are cheerful.",
        answer_template="I {agreement_scale}.",
        dimensions={"full_frequency_weights":full_frequency_weights,
                     "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":14,
                      "Original": 'I can usually work out when people are cheerful.'
        },
        **kwargs,
    )


qs14 = split_question(BESQ14,
                      index=["full_frequency_weights"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ14().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ14', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ16: I can usually realize quickly when a friend is angry.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ16(QMNLI):
  """
    BESQ16: I can usually realize quickly when a friend is angry.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I can {full_frequency_weights} realize quickly when a friend is angry.",
        answer_template="I {agreement_scale}.",
        dimensions={"full_frequency_weights":full_frequency_weights,
                    "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":16,
                      "Original": 'I can usually realize quickly when a friend is angry.'
        },
        **kwargs,
    )


qs16 = split_question(BESQ16,
                      index=["full_frequency_weights"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ16().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ16', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ19: I am not usually aware of my friend’s feelings.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ19(QMNLI):
  """
    BESQ19: I am not usually aware of my friend’s feelings.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I am {full_frequency_weights} aware of my friend’s feelings.",
        answer_template="I {agreement_scale}.",
        dimensions={"full_frequency_weights":full_frequency_weights,
                     "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":19,
                      "Original": 'I am usually aware of my friend’s feelings.' #REVERSED
        },
        **kwargs,
    )



qs19 = split_question(BESQ19,
                      index=["full_frequency_weights"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ19().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ19', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

#### BESQ20: I have trouble figuring out when my friends are happy.

In [ ]:
kw_pos = ['often', 'usually','frequently','always','very frequently']
kw_neg = ['seldom', 'rarely','never','very rarely']
dict_objective = dict_pos_neg(kw_pos,kw_neg,1)

class BESQ20(QMNLI):
  """
    BESQ20: I have trouble figuring out when my friends are happy.
  """
  def __init__(self, **kwargs):
    super().__init__(
        context_template="I {full_frequency_weights} figure out when my friends are happy.",
        answer_template="I {agreement_scale}.",
        dimensions={"full_frequency_weights":full_frequency_weights,
                  "agreement_scale":agreement_weights,
        },
        descriptor = {"Questionnair":"BES",
                      "Factor":"CE",
                      "Ordinal":20,
                      "Original": 'I usually figure out when my friends are happy.' # REVERSED
        },
        **kwargs,
    )


qs20 = split_question(BESQ20,
                      index=["full_frequency_weights"],
                      scales=['agreement_scale'],
                      softmax=[False, True],
                      filters={
                          "unfiltered":{},
                          "positiveonly":BESQ20().get_filter_for_postive_keywords()
                          # "W":{"gender":kw_short_female_plural},
                          #"M":{"gender":kw_short_male_plural}},
                      },

                      )
# q = qs[4]
# binary_grouping=True
# q.run(mnli).report(binary_grouping=binary_grouping)
# df = linguistic_acceptabilities(q, q._index, q._scale, 'BESQ20', 'student_id', output_path=Path(''),binary_grouping=binary_grouping)
# cols = ['semantic_similarity', 'cola_score', 'silhouette_score']
# df[cols].mean(axis=0)

# Run Questionnaires on models

## Utility functions

In [ ]:
def question_attributes(q):
    score = {}
    score['questionnair']=q._descriptor['Questionnair']
    score['factor']=q._descriptor['Factor']
    score['ordinal']=q._descriptor['Ordinal']
    score['scale']=q._descriptor['scale']
    score['index']=q._descriptor['index']
    score['filter']=q._descriptor['filter']
    score['softmax'] = q._descriptor['softmax']
    score["original"] = q._descriptor['Original']
    score['Q'] = f"{score['questionnair']}{score['factor']}{score['ordinal']}"
    score['context_template'] = q._context_template
    score['answer_template'] = q._answer_template
    score['dimensions'] = q._dimensions
    score['model'] = q.model.model_identifier if q.model else ""
    return score

def get_question_features(q, student_id='student_id', output_path=Path(''), save_to_file=False):
    score = question_attributes(q)
    score['mean_score'] = q.mean_score()
    index= q._index
    scale= q._scale
    linguistic_df = linguistic_acceptabilities(q, index=index, scale=scale,question_name=score['Q'], student_id=student_id,
                                               output_path=output_path, save_to_file=save_to_file)
    row = linguistic_df[['cola_score','silhouette_score']].mean(axis=0)
    row_dict = dict(row)
    row_dict['semantic_similarity'] = linguistic_df['semantic_similarity'].quantile(0.75)
    score = score | row_dict
    return score

def extract_epoch(model_path):
    if 'epoch-' in model_path.name:
        i = model_path.name.find('epoch-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('epoch-'):j])
        else:
            epoch = int(model_path.name[i+len('epoch-'):])

    elif 'checkpoint-' in model_path.name:
        i = model_path.name.find('checkpoint-')
        j = model_path.name.find('_', i)
        if j > 0:
            epoch = int(model_path.name[i+len('checkpoint-'):j])
        else:
            epoch = int(model_path.name[i+len('checkpoint-'):])
    else:
        epoch = 0
    return epoch

def extract_run(model_path):
    try:
        if 'run' in model_path.name:
            for part in model_path.name.split('_'):
                if 'run' in part:
                    return int(part.replace('run', ''))
        else:
            return -1
    except Exception as e:
        print(e)
        return -1

import json

def get_mnli_score(checkpoint_path):
    mnli_score_path = checkpoint_path / 'all_results.json'
    if not mnli_score_path.exists():
        mnli_score_path = checkpoint_path.parent / (checkpoint_path.name + '_mnli_eval') / 'all_results.json'
    if mnli_score_path.exists():
        with open(mnli_score_path) as f:
            return json.load(f)["eval_accuracy"]
    else:
        return -1


def run_questions(questions, mnli_checkpoint, train_process, fintune_dataset, q_range=[5, 0]):
    rows = []
    checkpoint = Path(mnli_checkpoint.model_identifier)
    for q_raw in tqdm(questions):
        T = time.time()
        q = q_raw.run(mnli_checkpoint)
        T = time.time()
        score = get_question_features(q)
        score['epoch'] = extract_epoch(checkpoint)
        score['train_process'] = train_process
        score['dataset'] = fintune_dataset
        score['run'] = extract_run(checkpoint.parent)
        score['mnli_score'] = get_mnli_score(checkpoint)
        score['range'] = (q._weights_flat.min(), q._weights_flat.max())
        score['ASI_score'] = np.interp(score['mean_score'], [q._weights_flat.min(), q._weights_flat.max()], q_range)
        rows.append(score)
        gc.collect()
        torch.cuda.empty_cache()
    return rows


def calc_scores(questions, checkpoint, output_path, train_process, fintune_dataset, q_range=[5, 0]):
    fix_config(checkpoint)
    mnli_checkpoint = pipeline("zero-shot-classification", str(checkpoint), device=device)
    mnli_checkpoint.model_identifier = str(checkpoint)
    rows = run_questions(questions, mnli_checkpoint, train_process, fintune_dataset=fintune_dataset, q_range=q_range)
    return rows

def add_epochs_to_rows(rows, mlm_epoch, mnli_checkpoint):
    for score in rows:
        score['mlm_epoch'] = mlm_epoch
        score['mnli_checkpoint'] = mnli_checkpoint
    return rows


def write_to_csv(rows, output_path):
    old_score_hostile_df = pd.DataFrame(rows)
    if output_path.exists():
        old_score_hostile_df.to_csv(output_path, index=False, header=None, mode='a')
    else:
        old_score_hostile_df.to_csv(output_path, index=False)

def fix_config(checkpoint):
    if checkpoint.exists():
        with open(checkpoint / 'config.json') as f:
            d1 = json.load(f)
        d1['id2label'] = {'0': 'entailment', '1': 'neutral', '2': 'contradiction'}
        d1['label2id'] = {'contradiction': 2, 'entailment': 0, 'neutral': 1}
        with open(checkpoint / 'config.json', 'w') as f:
            json.dump(d1, f)
    else:
        print(checkpoint, '#### Not exists ####')

def calc_for_all_models(Qs, q_range= [5, 0]):
    all_rows = []
    for p in tqdm(mnli_pipelines):
        print(p)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            rows = calc_scores(Qs, Path(p),  Path(p), '->'.join(['base']), 'hostile',
                               use_base_model=False, q_range=q_range)
            rows = add_epochs_to_rows(rows, 0, 0)
            all_rows += rows
    return pd.DataFrame(all_rows)

## Run Questions

In [ ]:
result_path = Path('results/')
if not result_path.exists():
    os.makedirs(result_path)

In [ ]:
mnli_pipelines = [
    'typeform/distilbert-base-uncased-mnli',
    'typeform/mobilebert-uncased-mnli',
    'cross-encoder/nli-roberta-base',
    'cross-encoder/nli-deberta-base',
    'cross-encoder/nli-distilroberta-base',
    'cross-encoder/nli-MiniLM2-L6-H768',
    'navteca/bart-large-mnli',
    'digitalepidemiologylab/covid-twitter-bert-v2-mnli',
    'joeddav/bart-large-mnli-yahoo-answers',
    'Narsil/deberta-large-mnli-zero-cls',
    'microsoft/deberta-large-mnli',
    'microsoft/deberta-base-mnli',
    'Alireza1044/albert-base-v2-mnli',
    'yoshitomo-matsubara/bert-large-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli',
    'yoshitomo-matsubara/bert-base-uncased-mnli_from_bert-large-uncased-mnli',
    'valhalla/distilbart-mnli-12-6',
]



In [ ]:

questions = qs1+qs2+qs3+qs4+qs5+qs6+qs7+qs8+qs9+qs10+qs11+qs12+qs13+qs14+qs15+qs16+qs17+qs18+qs19+qs20
from collections import defaultdict
update = True

output_path = result_path / f'bes_mnli_all_models_v1.csv'
# [str(a) for a in Path('mnli_models/').glob('*_mnli')]
pipelines = mnli_pipelines + [str(a) for a in Path('/dt/puzis/cnalab/maor/mnli_models/').glob('*_mnli')]

if output_path.exists():
    temp_df = pd.read_csv(output_path)
    indexes = temp_df.groupby(['model', 'Q']).count().index.values
    used_models = defaultdict(set)
    for k, v in indexes:
        used_models[k].add(v)
else:
    used_models = {}


for p in tqdm(pipelines):
    print(p)
    if get_mnli_score(Path(p)) < 0.7 and p not in mnli_pipelines:
        print('Skip:', p)
        continue
    with warnings.catch_warnings():
        try:
            warnings.simplefilter("ignore")
            if p in used_models and not update:
                pipline_questions = []
                for q in questions:
                    if question_attributes(q)['Q'] not in used_models[p]:
                        pipline_questions.append(q)
                    else:
                        print('skip', p, question_attributes(q)['Q'])
            else:
                pipline_questions = questions
            rows = calc_scores(pipline_questions, Path(p),  output_path, '->'.join(['base']), 'hostile',)
            rows = add_epochs_to_rows(rows, 0, 0)
            print("made file")
            write_to_csv(rows, output_path)
            gc.collect()
            torch.cuda.empty_cache()
        except Exception as e:
            print(e)


df = pd.read_csv(output_path)
df = df.drop_duplicates(subset=['filter','softmax','model','Q'], keep='last')
df.to_csv(output_path, index=False)

# Validations

In [ ]:
def load_results(csv_path, softmax, positiveonly, value='BES_score', index='model'):
    df = pd.read_csv(csv_path)
    df['model'] = df['model'].str.replace('/dt/puzis/cnalab/maor/', '')
    if df['softmax'].isna().sum() > 0:
        softmax_filter = df['softmax'].isna()
    else:
        softmax_filter = df['softmax'] == ''
    if softmax:
        df = df[df['softmax'] == str(softmax)]
    else:
        df = df[softmax_filter]
    if value != 'silhouette_score':
        pass
    else:
        df = df[df['silhouette_score'] > -1]
    if positiveonly:
        df = df[df['filter']=="positiveonly"]
    else:
        df = df[df['filter']=="unfiltered"]
    results_df = pd.pivot_table(df, values=value, index=index, columns='Q', aggfunc='mean')
    return results_df

## Content Validity

In [ ]:
softmax_bes=['index', 'agreement_scale']
all_filters = [['full_frequency_weights', 'agreement_scale'],['partial_frequency_weights0', 'agreement_scale'],
              ['partial_frequency_weights1', 'agreement_scale'],['partial_frequency_weights2', 'agreement_scale'],
              ['partial_frequency_weights3', 'agreement_scale'],['partial_frequency_weights4', 'agreement_scale']]
positiveonly=True

lr = 2e-7
bes_factors = ['AE','CE']
q_path = result_path / f'bes_mnli_all_models_v1.csv'





### Semantic Validation

In [ ]:
cols = ['semantic_similarity', 'cola_score', 'silhouette_score']

results = []
for softmax_filter in all_filters:
    q_res = [load_results(q_path,softmax=softmax_filter,positiveonly=False, value=v).mean(axis=0) for v in cols]
    results.append(pd.concat(q_res, axis=1))

liguestic_acceptability_df = pd.concat(results, axis=0)
liguestic_acceptability_df.columns=['semantic_similarity', 'cola_score', 'silhouette_score']
liguestic_acceptability_df.to_csv(result_path / 'liguestic_acceptability.csv', index=False)
liguestic_acceptability_df

In [ ]:
liguestic_acceptability_df.mean()

In [ ]:
liguestic_acceptability_df.std()

### Internal Consistency

In [ ]:
def get_factor_sub_features(factor, data_df):
    factor = factor if isinstance(factor, list) else [factor]
    feature_subset = []
    for subset in factor:
        for c in data_df.columns:
            if str(subset) in c:
#                 if c[c.find(subset):].replace(subset, '').isnumeric():
                feature_subset.append(c)
    return list(set(feature_subset))

In [ ]:
value='mean_score'

results = []
# for softmax_filter in [softmax_soc, softmax_gad]:
for softmax_filter in all_filters:
    results.append(load_results(q_path,softmax=softmax_filter,positiveonly=positiveonly, value=value))

data_df = pd.concat(results, axis=1)

print('Cronbach Alpha:')
# for subset in soc_factors + gad_factors + phq_factors:
for subset in bes_factors:
    feature_subset = get_factor_sub_features(subset, data_df)
    alpha = pg.cronbach_alpha(data=data_df[feature_subset])
    print(f'{subset}, Alpha:, {alpha}')

bes_feature_subset = get_factor_sub_features(bes_factors, data_df)
alpha = pg.cronbach_alpha(data=data_df[bes_feature_subset])
print(f'bes, Alpha:, {alpha}')

In [ ]:
data_df[feature_subset]

In [ ]:
for subset in ['AE','CE']:
    feature_subset = [c for c in data_df.columns if subset in c]
#     subset_df = data_df[feature_subset].drop(subset, axis=1)
    subset_df = data_df[feature_subset]
    alpha = pg.cronbach_alpha(data=subset_df)
    print(subset, 'Alpha:', alpha)
    for feature in subset_df.columns:
        sub = [c for c in subset_df.columns if c != feature]
        alpha = pg.cronbach_alpha(data=subset_df[sub])
        print('without:', feature, 'Alpha:', alpha)


## Construct Validity

In [ ]:
value='mean_score'

results = []
# for softmax_filter in [softmax_soc, softmax_gad]:
for softmax_filter in all_filters:
    results.append(load_results(q_path,softmax=softmax_filter,positiveonly=positiveonly, value=value))

data_df = pd.concat(results, axis=1)

filterd_df = pd.DataFrame()
# for factor in gad_factors + phq_factors + soc_factors:
for factor in bes_factors:
    feature_subset = get_factor_sub_features([factor], data_df)
    filterd_df[factor] = data_df[feature_subset].mean(axis=1)

# soc_feature_subset = get_factor_sub_features(soc_factors, data_df)
# filterd_df['SOC13'] = data_df[soc_feature_subset].mean(axis=1)
bes_feature_subset = get_factor_sub_features(bes_factors, data_df)
filterd_df['BES'] = data_df[bes_feature_subset].mean(axis=1)

filterd_df.rcorr(method='spearman')